# Supporting Materials - MEA Analysis

Notebook for manuscript supporting material. Analyses:

1. **Raster plots** - spike rasters from Maxwell `.raw.h5` files
2. **ISI & bursting** - inter-spike intervals and burst metrics (top-N channels)
3. **STTC connectivity** - pairwise Spike Timing Tiling Coefficient (top-N electrodes)

**Data:** Maxwell HDMEA recordings as `.raw.h5`. Spike times at `recordings/rec0000/well000/spikes` (20 kHz).

**Dependencies:** numpy, matplotlib, h5py, pandas

In [ ]:
import os
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from matplotlib.colors import TwoSlopeNorm

plt.rcParams['font.size'] = 10

# --- paths (edit these) ---
RECORDING_DIR = './data/recordings'   # folder with *.raw.h5 files
OUTPUT_DIR = './output'

SAMPLING_RATE_HZ = 20_000
SPIKE_DS_PATH = 'recordings/rec0000/well000/spikes'
MAPPING_DS_PATH = 'wells/well000/rec0000/settings/mapping'

RASTER_ALL_DIR = os.path.join(OUTPUT_DIR, 'raster_plots', 'raster_all')
RASTER_TOP25_DIR = os.path.join(OUTPUT_DIR, 'raster_plots', 'raster_top25')
ISI_BURST_DIR = os.path.join(OUTPUT_DIR, 'isi_bursting')
STTC_DIR = os.path.join(OUTPUT_DIR, 'sttc_connectivity')

for d in [RASTER_ALL_DIR, RASTER_TOP25_DIR, ISI_BURST_DIR, STTC_DIR]:
    os.makedirs(d, exist_ok=True)

print('Recording dir:', RECORDING_DIR)
print('Output dir:', OUTPUT_DIR)


## Shared helpers

Functions used by raster plots, ISI/bursting, and STTC sections.

In [ ]:
def get_most_spiking_channels(file_path, n_channels=25):
    with h5py.File(file_path, 'r') as f:
        if SPIKE_DS_PATH not in f:
            print('Spike data not found:', SPIKE_DS_PATH)
            return None, None
        spikes = f[SPIKE_DS_PATH][:]
        ch_ids, counts = np.unique(spikes['channel'], return_counts=True)
        ranked = sorted(zip(ch_ids.astype(int), counts.astype(int)), key=lambda x: -x[1])
        top = ranked[:n_channels]
        return [ch for ch, _ in top], [c for _, c in top]


def extract_channel_coordinates(file_path, channel_ids):
    coords = []
    with h5py.File(file_path, 'r') as f:
        mapping = f[MAPPING_DS_PATH][:]
        lookup = {int(row['channel']): (float(row['x']), float(row['y'])) for row in mapping}
        for ch in channel_ids:
            if ch in lookup:
                coords.append(list(lookup[ch]))
            else:
                coords.append([np.nan, np.nan])
    return np.array(coords)


def extract_spike_trains_top100(file_path, channel_ids):
    ch_set = set(channel_ids)
    idx_map = {ch: i for i, ch in enumerate(channel_ids)}
    trains = [[] for _ in range(len(channel_ids))]
    with h5py.File(file_path, 'r') as f:
        for row in f[SPIKE_DS_PATH][:]:
            ch = int(row['channel'])
            if ch in ch_set:
                trains[idx_map[ch]].append(int(row['frameno']))
    for i in range(len(trains)):
        trains[i].sort()
    return trains


## Raster plots

300 s window centered on the midpoint between first and last spike (same as `mid300s` in ISI/bursting below).

- **All channels:** any channel with >5 spikes in the window
- **Top 25:** channels ranked by spike count within the window

In [ ]:
window_duration = 300  # seconds
sampling_rate = SAMPLING_RATE_HZ

h5_files = sorted([f for f in os.listdir(RECORDING_DIR) if f.endswith('.h5')])
conditions = [f.replace('.raw.h5', '').replace('.h5', '') for f in h5_files]

for file, condition in zip(h5_files, conditions):
    print('Raster (all channels):', condition)
    file_path = os.path.join(RECORDING_DIR, file)

    channel_framenos = [[] for _ in range(1024)]
    all_framenos = []

    with h5py.File(file_path, 'r') as f:
        for spike_entry in f[SPIKE_DS_PATH][:]:
            ch = int(spike_entry['channel'])
            fn = int(spike_entry['frameno'])
            if 0 <= ch < 1024:
                channel_framenos[ch].append(fn)
                all_framenos.append(fn)

    all_framenos = np.array(all_framenos)
    first_spike = np.min(all_framenos)
    last_spike = np.max(all_framenos)
    middle = first_spike + (last_spike - first_spike) / 2
    window_frames = int(window_duration * sampling_rate)
    win_start = int(middle - window_frames / 2)
    win_end = win_start + window_frames

    fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
    n_plotted = 0
    for ch_id in range(1024):
        spikes = np.array(channel_framenos[ch_id])
        spikes = spikes[(spikes >= win_start) & (spikes <= win_end)]
        if len(spikes) > 5:
            times = (spikes - win_start) / sampling_rate
            ax.eventplot(times, lineoffsets=ch_id, linelengths=0.8, colors='black')
            n_plotted += 1

    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('Channel ID')
    ax.set_title(f'Raster - {condition} (>5 spikes, 300s window)')
    ax.set_xlim(0, window_duration)
    out_path = os.path.join(RASTER_ALL_DIR, f'raster_{condition}.png')
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print('  Saved:', out_path, f'({n_plotted} channels)')

print('Done - all-channel rasters.')


In [ ]:
window_duration = 300
sampling_rate = SAMPLING_RATE_HZ

h5_files = sorted([f for f in os.listdir(RECORDING_DIR) if f.endswith('.h5')])
conditions = [f.replace('.raw.h5', '').replace('.h5', '') for f in h5_files]

for file, condition in zip(h5_files, conditions):
    print('Raster (top 25):', condition)
    file_path = os.path.join(RECORDING_DIR, file)

    channel_framenos = [[] for _ in range(1024)]
    all_framenos = []

    with h5py.File(file_path, 'r') as f:
        for spike_entry in f[SPIKE_DS_PATH][:]:
            ch = int(spike_entry['channel'])
            fn = int(spike_entry['frameno'])
            if 0 <= ch < 1024:
                channel_framenos[ch].append(fn)
                all_framenos.append(fn)

    all_framenos = np.array(all_framenos)
    first_spike = np.min(all_framenos)
    last_spike = np.max(all_framenos)
    middle = first_spike + (last_spike - first_spike) / 2
    window_frames = int(window_duration * sampling_rate)
    win_start = int(middle - window_frames / 2)
    win_end = win_start + window_frames

    counts_in_window = []
    for ch_id in range(1024):
        spikes = np.array(channel_framenos[ch_id])
        spikes = spikes[(spikes >= win_start) & (spikes <= win_end)]
        if len(spikes) >= 5:
            counts_in_window.append((ch_id, len(spikes)))
    counts_in_window.sort(key=lambda x: -x[1])
    top_25 = counts_in_window[:25]

    fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
    for y_pos, (ch_id, _) in enumerate(top_25):
        spikes = np.array(channel_framenos[ch_id])
        spikes = spikes[(spikes >= win_start) & (spikes <= win_end)]
        times = (spikes - win_start) / sampling_rate
        ax.eventplot(times, lineoffsets=y_pos, linelengths=0.8, colors='black', linewidth=0.7)

    ax.set_yticks(range(len(top_25)))
    ax.set_yticklabels([str(ch) for ch, _ in top_25], fontsize=9)
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('Channel ID (sorted by spike count)')
    ax.set_title(f'Raster - {condition} (Top 25, 300s window)')
    ax.set_xlim(0, window_duration)
    out_path = os.path.join(RASTER_TOP25_DIR, f'raster_top25_{condition}.png')
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print('  Saved:', out_path)

print('Done - top-25 rasters.')


## ISI and bursting analysis

Top-N channels ranked by spike count over the **full** recording. Metrics for epochs `full` and `mid300s` (300 s mid-window).

**Burst parameters (editable in code):** `BURST_INTRABURST_ISI_MAX_S=0.1`, `BURST_MIN_SPIKES=3`, `TOP_N_CHANNELS=25`

**ISI metrics:** mean/median/std ISI, CV, spike rate

**Burst metrics:** mean/median burst duration, spikes/burst, burst frequency, fraction in bursts, total bursts across top-N

In [ ]:
TOP_N_CHANNELS = 25
MIDDLE_WINDOW_SEC = 300.0
MIN_SPIKES_ISI_FULL = 50
MIN_SPIKES_ISI_MID300S = 20
MIN_SPIKES_BURST_FULL = MIN_SPIKES_ISI_FULL
MIN_SPIKES_BURST_MID300S = MIN_SPIKES_ISI_MID300S

BURST_INTRABURST_ISI_MAX_S = 0.1
BURST_MIN_SPIKES = 3
BURST_MIN_DURATION_S = 0.0
BURST_MAX_BURST_DURATION_S = None


def gap_split_segments(t_sec, thr_s):
    t_sec = np.asarray(t_sec, dtype=float)
    if t_sec.size == 0:
        return []
    segs, s = [], 0
    for i in range(1, t_sec.size):
        if t_sec[i] - t_sec[i - 1] > thr_s:
            segs.append(t_sec[s:i])
            s = i
    segs.append(t_sec[s:])
    return segs


def segments_respecting_max_span(seg, thr_s, min_sp, min_du_s, max_span_s):
    seg = np.asarray(seg, dtype=float)
    n = seg.size
    bursts = []
    if max_span_s is None:
        if n >= min_sp and seg[-1] - seg[0] >= min_du_s:
            return [seg.copy()]
        return bursts
    i = 0
    while i <= n - min_sp:
        j_end = -1
        for j in range(i + min_sp - 1, n):
            if seg[j] - seg[i] > max_span_s:
                break
            if not np.all(np.diff(seg[i:j + 1]) <= thr_s):
                break
            j_end = j
        if j_end < 0:
            i += 1
            continue
        cand = seg[i:j_end + 1]
        if cand[-1] - cand[0] >= min_du_s:
            bursts.append(cand.copy())
        i = j_end + 1
    return bursts


def burst_intervals_sorted(t_sec):
    all_bursts = []
    for run in gap_split_segments(t_sec, BURST_INTRABURST_ISI_MAX_S):
        run = np.sort(run)
        if run.size == 0:
            continue
        all_bursts.extend(segments_respecting_max_span(
            run, BURST_INTRABURST_ISI_MAX_S, BURST_MIN_SPIKES,
            BURST_MIN_DURATION_S, BURST_MAX_BURST_DURATION_S))
    return all_bursts


def compute_burst_summaries(intervals_list, denom_sec, n_spikes):
    if not intervals_list:
        frac = np.nan if n_spikes <= 0 else 0.0
        fq = 0.0 / denom_sec if denom_sec and denom_sec > 0 else np.nan
        return {
            'n_bursts': 0, 'burst_freq_hz': fq,
            'burst_freq_per_min': fq * 60 if np.isfinite(fq) else np.nan,
            'mean_burst_duration_s': np.nan, 'median_burst_duration_s': np.nan,
            'mean_spikes_per_burst': np.nan, 'median_spikes_per_burst': np.nan,
            'frac_spikes_in_bursts': frac,
        }
    durs = np.array([seg[-1] - seg[0] for seg in intervals_list])
    sns = np.array([len(seg) for seg in intervals_list], dtype=float)
    n_b = len(intervals_list)
    fq = n_b / denom_sec if denom_sec and denom_sec > 0 else np.nan
    return {
        'n_bursts': int(n_b),
        'burst_freq_hz': fq,
        'burst_freq_per_min': n_b / (denom_sec / 60.0) if denom_sec and denom_sec > 0 else np.nan,
        'mean_burst_duration_s': float(np.mean(durs)),
        'median_burst_duration_s': float(np.median(durs)),
        'mean_spikes_per_burst': float(np.mean(sns)),
        'median_spikes_per_burst': float(np.median(sns)),
        'frac_spikes_in_bursts': float(np.sum(sns)) / max(n_spikes, 1),
    }


def load_spikes_per_channel(h5_path):
    chan_frames = defaultdict(list)
    gmin, gmax = None, None
    with h5py.File(h5_path, 'r') as f:
        if SPIKE_DS_PATH not in f:
            return None, None, None
        for row in f[SPIKE_DS_PATH][:]:
            ch = int(row['channel'])
            fn = int(row['frameno'])
            chan_frames[ch].append(fn)
            gmin = fn if gmin is None else min(gmin, fn)
            gmax = fn if gmax is None else max(gmax, fn)
    for ch in chan_frames:
        chan_frames[ch] = np.array(chan_frames[ch], dtype=np.int64)
    dur_full = (gmax - gmin) / SAMPLING_RATE_HZ
    half_w = int((MIDDLE_WINDOW_SEC / 2.0) * SAMPLING_RATE_HZ)
    mid_fr = gmin + int((gmax - gmin) / 2)
    win_lo = int(mid_fr - half_w)
    win_hi = win_lo + int(MIDDLE_WINDOW_SEC * SAMPLING_RATE_HZ)
    return chan_frames, dur_full, (win_lo, win_hi)


def agg_recording_metric(long_df, value_cols):
    rows = []
    for ep in ('full', 'mid300s'):
        sub = long_df.loc[long_df['epoch'] == ep]
        gb = sub.groupby('condition', as_index=False)[value_cols].mean(numeric_only=True)
        gb.insert(1, 'epoch', ep)
        rows.append(gb)
    return pd.concat(rows, ignore_index=True)


In [ ]:
isi_long_rows = []
burst_long_rows = []

h5_files = sorted([f for f in os.listdir(RECORDING_DIR) if f.endswith('.h5')])

for file in h5_files:
    condition = file.replace('.raw.h5', '').replace('.h5', '')
    h5_path = os.path.join(RECORDING_DIR, file)
    print('ISI/burst:', condition)

    ch_fm, dur_full, win_pair = load_spikes_per_channel(h5_path)
    if ch_fm is None:
        print('  skip - no spike data')
        continue

    ranked = sorted(((ch, len(fr)) for ch, fr in ch_fm.items()), key=lambda x: -x[1])
    top_ids = [ch for ch, _ in ranked[:TOP_N_CHANNELS]]
    win_lo, win_hi = win_pair

    for ch in top_ids:
        fr_all = np.sort(np.asarray(ch_fm[ch]))
        fr_mid = fr_all[(fr_all >= win_lo) & (fr_all <= win_hi)]

        for epoch_label, framenos, denom_s, gate_isi, gate_burst in [
            ('full', fr_all, dur_full, MIN_SPIKES_ISI_FULL, MIN_SPIKES_BURST_FULL),
            ('mid300s', fr_mid, MIDDLE_WINDOW_SEC, MIN_SPIKES_ISI_MID300S, MIN_SPIKES_BURST_MID300S),
        ]:
            t = np.sort(framenos.astype(float) / SAMPLING_RATE_HZ)

            mean_is = med_is = cv_is = std_is = spike_hz = np.nan
            if t.size >= 2:
                isi = np.diff(t)
                if t.size >= gate_isi:
                    mean_is = float(np.mean(isi))
                    med_is = float(np.median(isi))
                    std_is = float(np.std(isi, ddof=1))
                    cv_is = std_is / mean_is if mean_is > 0 else np.nan
                if denom_s and denom_s > 0:
                    spike_hz = t.size / denom_s

            isi_long_rows.append({
                'condition': condition, 'epoch': epoch_label, 'channel_id': ch,
                'n_spikes': int(t.size), 'spike_rate_hz_over_epoch': spike_hz,
                'mean_isi_s': mean_is, 'median_isi_s': med_is,
                'std_isi_s': std_is, 'cv_isi': cv_is,
            })

            bursts = burst_intervals_sorted(t)
            burst_stats = compute_burst_summaries(
                bursts, denom_s if denom_s and denom_s > 0 else None, int(t.size))
            if t.size < gate_burst:
                for k in burst_stats:
                    burst_stats[k] = np.nan

            burst_long_rows.append({
                'condition': condition, 'epoch': epoch_label, 'channel_id': ch,
                'n_spikes': int(t.size), **burst_stats,
            })

isi_long_df = pd.DataFrame(isi_long_rows)
burst_long_df = pd.DataFrame(burst_long_rows)

isi_long_df.to_csv(os.path.join(ISI_BURST_DIR, 'channel_isi_long.csv'), index=False)
burst_long_df.to_csv(os.path.join(ISI_BURST_DIR, 'channel_burst_long.csv'), index=False)
print('Saved channel-level CSVs')

isi_met = ['spike_rate_hz_over_epoch', 'mean_isi_s', 'median_isi_s', 'cv_isi', 'std_isi_s']
isi_summ = agg_recording_metric(isi_long_df, [c for c in isi_met if c in isi_long_df.columns])
isi_summ.to_csv(os.path.join(ISI_BURST_DIR, 'recording_isi_summaries.csv'), index=False)

burst_met = [c for c in burst_long_df.columns if c not in {'condition', 'epoch', 'channel_id', 'n_spikes'}]
b_summ = agg_recording_metric(burst_long_df, burst_met)

# total burst count and recording-level frequency
tot = burst_long_df.groupby(['condition', 'epoch'], as_index=False).agg(
    total_n_bursts_across_topN=('n_bursts', 'sum'))
# attach epoch duration for frequency
dur_lookup = {}
for file in h5_files:
    cond = file.replace('.raw.h5', '').replace('.h5', '')
    _, dur_full, _ = load_spikes_per_channel(os.path.join(RECORDING_DIR, file))
    dur_lookup[(cond, 'full')] = dur_full
    dur_lookup[(cond, 'mid300s')] = MIDDLE_WINDOW_SEC
tot['denom_seconds'] = tot.apply(lambda r: dur_lookup.get((r['condition'], r['epoch']), np.nan), axis=1)
tot['recording_total_burst_freq_hz'] = tot['total_n_bursts_across_topN'] / tot['denom_seconds']
tot['recording_total_burst_freq_per_min'] = tot['recording_total_burst_freq_hz'] * 60

b_summ = pd.merge(b_summ, tot, on=['condition', 'epoch'], how='left')
b_summ.to_csv(os.path.join(ISI_BURST_DIR, 'recording_burst_summaries.csv'), index=False)

print('Saved recording summaries to', ISI_BURST_DIR)


## STTC connectivity (top-N electrodes)

Pairwise **Spike Timing Tiling Coefficient** (Cutts & Eglen, 2014) for the top-N most active channels.

- **`delta_ms`** - coincidence window (default 10 ms)
- **`STTC_THRESHOLD`** - pairs with STTC > threshold counted as significant (default 0.1)
- **`mean_connectivity`** - mean STTC over all pairs
- **`connectivity_score`** - mean STTC over significant pairs only

Outputs per recording: matrix heatmap, spatial heatmap, row in `connectivity_scores.csv`.

In [ ]:
# STTC parameters
TOP_N_STTC = 100
DELTA_MS = 10
STTC_THRESHOLD = 0.1


def calculate_sttc(spike_times_1, spike_times_2, delta=0.01):
    # delta in seconds (e.g. 0.01 = 10 ms)
    if len(spike_times_1) == 0 or len(spike_times_2) == 0:
        return 0.0

    P_B = 0
    for t1 in spike_times_1:
        P_B += np.sum((spike_times_2 >= t1 - delta) & (spike_times_2 <= t1 + delta))

    T_A = 0
    for index1 in range(len(spike_times_1)):
        if index1 > 0:
            dt = spike_times_1[index1] - spike_times_1[index1 - 1]
            T_A += dt if dt < 2 * delta else 2 * delta

    P_B = P_B / len(spike_times_2)
    T_A = T_A / (spike_times_1[-1] - spike_times_1[0])

    P_A = 0
    for t2 in spike_times_2:
        P_A += np.sum((spike_times_1 >= t2 - delta) & (spike_times_1 <= t2 + delta))

    T_B = 0
    for index2 in range(len(spike_times_2)):
        if index2 > 0:
            dt = spike_times_2[index2] - spike_times_2[index2 - 1]
            T_B += dt if dt < 2 * delta else 2 * delta

    P_A = P_A / len(spike_times_1)
    T_B = T_B / (spike_times_2[-1] - spike_times_2[0])

    sttc = 0.5 * ((P_A - T_B) / (1 - P_A * T_B + 1e-10)) + 0.5 * ((P_B - T_A) / (1 - P_B * T_A + 1e-10))
    return sttc


def calculate_connectivity_matrix_top100(channel_framenos, sampling_rate=20000, delta_ms=10):
    n = len(channel_framenos)
    mat = np.zeros((n, n))
    print(f'  STTC matrix: {n} channels, delta={delta_ms} ms')
    for i in range(n):
        if len(channel_framenos[i]) < 5:
            continue
        for j in range(n):
            if i == j or len(channel_framenos[j]) < 5:
                continue
            ti = np.array(channel_framenos[i]) / sampling_rate
            tj = np.array(channel_framenos[j]) / sampling_rate
            mat[i, j] = calculate_sttc(ti, tj, delta=delta_ms / 1000)
        if (i + 1) % 20 == 0:
            print(f'    {i + 1}/{n} channels done')
    return mat


def calculate_network_connectivity_score(connectivity_matrix, threshold=0.1):
    mat = connectivity_matrix.copy()
    np.fill_diagonal(mat, 0)
    all_conn = mat[mat != 0]
    sig = all_conn[all_conn > threshold]
    return {
        'mean_connectivity': float(np.mean(all_conn)) if len(all_conn) else 0,
        'connectivity_score': float(np.mean(sig)) if len(sig) else 0,
        'network_density': len(sig) / len(all_conn) if len(all_conn) else 0,
        'num_significant_connections': len(sig),
        'num_total_connections': len(all_conn),
        'threshold': threshold,
    }


def plot_connectivity_matrix_heatmap(connectivity_matrix, condition_name, output_folder):
    fig, ax = plt.subplots(figsize=(10, 9), dpi=300)
    # show 25 most active rows/cols for readability
    top_idx = np.argsort(np.sum(np.abs(connectivity_matrix), axis=1))[-25:]
    im = ax.imshow(connectivity_matrix[np.ix_(top_idx, top_idx)],
                   cmap='RdBu_r', aspect='auto', vmin=-0.5, vmax=0.5)
    ax.set_xlabel('Target channel index')
    ax.set_ylabel('Source channel index')
    ax.set_title(f'{condition_name} - STTC matrix (25 most connected)')
    plt.colorbar(im, ax=ax, label='STTC')
    plt.tight_layout()
    path = os.path.join(output_folder, f'connectivity_matrix_{condition_name}.png')
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.close()
    print('  Saved:', path)


def plot_connectivity_heatmap_spatial_centered(connectivity_matrix, channel_coords,
                                               condition_name, output_folder, threshold=0.1):
    fig, axes = plt.subplots(1, 2, figsize=(16, 7), dpi=300)

    ax1 = axes[0]
    ax1.scatter(channel_coords[:, 0], channel_coords[:, 1],
                s=100, c='lightgray', edgecolors='black', linewidth=0.5)
    n = len(channel_coords)
    for i in range(n):
        for j in range(n):
            if i != j and connectivity_matrix[i, j] > threshold:
                ax1.plot([channel_coords[i, 0], channel_coords[j, 0]],
                         [channel_coords[i, 1], channel_coords[j, 1]],
                         'b-', alpha=min(1.0, connectivity_matrix[i, j] / 0.5) * 0.6, linewidth=1.5)
    ax1.set_xlabel('X (um)')
    ax1.set_ylabel('Y (um)')
    ax1.set_title(f'{condition_name} - connections STTC > {threshold}')
    ax1.set_aspect('equal')
    ax1.grid(True, alpha=0.3)

    ax2 = axes[1]
    in_strength = np.sum(connectivity_matrix, axis=0)
    vmax = max(abs(in_strength.min()), abs(in_strength.max()), 1e-9)
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    sc = ax2.scatter(channel_coords[:, 0], channel_coords[:, 1],
                     c=in_strength, s=150, cmap='RdBu_r', norm=norm,
                     edgecolors='black', linewidth=0.5)
    plt.colorbar(sc, ax=ax2, label='Sum incoming STTC')
    ax2.set_xlabel('X (um)')
    ax2.set_ylabel('Y (um)')
    ax2.set_title(f'{condition_name} - incoming STTC sum')
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(output_folder, f'connectivity_heatmap_spatial_{condition_name}.png')
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.close()
    print('  Saved:', path)


In [ ]:
network_scores = []

h5_files = sorted([f for f in os.listdir(RECORDING_DIR) if f.endswith('.h5')])

for file in h5_files:
    condition = file.replace('.raw.h5', '').replace('.h5', '')
    file_path = os.path.join(RECORDING_DIR, file)
    print('=' * 50)
    print('STTC:', condition)

    top_channels, _ = get_most_spiking_channels(file_path, n_channels=TOP_N_STTC)
    if top_channels is None:
        print('  skip')
        continue

    coords = extract_channel_coordinates(file_path, top_channels)
    trains = extract_spike_trains_top100(file_path, top_channels)

    conn_mat = calculate_connectivity_matrix_top100(trains, SAMPLING_RATE_HZ, DELTA_MS)

    plot_connectivity_matrix_heatmap(conn_mat, condition, STTC_DIR)
    plot_connectivity_heatmap_spatial_centered(conn_mat, coords, condition, STTC_DIR, STTC_THRESHOLD)

    score = calculate_network_connectivity_score(conn_mat, STTC_THRESHOLD)
    print(f'  mean_connectivity={score["mean_connectivity"]:.6f}')
    print(f'  connectivity_score (STTC>{STTC_THRESHOLD})={score["connectivity_score"]:.6f}')
    print(f'  significant pairs: {score["num_significant_connections"]}/{score["num_total_connections"]}')

    network_scores.append({
        'condition': condition,
        'file': file,
        **score,
    })

scores_df = pd.DataFrame(network_scores)
scores_path = os.path.join(STTC_DIR, 'connectivity_scores.csv')
scores_df.to_csv(scores_path, index=False)
print('Saved:', scores_path)
